# COMP9085 Week 3 Laboratory
## From dataset to evidence: building a reliable ML workflow

**Two hours.** Week 2 ended with a decision tree that scored 91.8 per cent.
This lab asks the question this course spends Week 3 on: **how do you know
that number means anything?**

You will measure how unstable a single train/test split is, run cross-validation
properly, build a leak-free pipeline and then deliberately break it to measure
the damage, tune a hyperparameter without touching the test set, and finish by
writing the protocol down so someone else could reproduce it.

**Dataset.** Wisconsin Diagnostic Breast Cancer (WDBC): 569 cases, 30 features,
212 malignant and 357 benign. Real benchmark data, used here for educational
machine-learning experimentation. **Nothing in this notebook is a clinical
diagnostic system.**

---

### Before you start

```
pip install numpy pandas scikit-learn matplotlib
```

Everything runs on a laptop in a couple of minutes. No GPU, no internet after
the first import.

### Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GroupShuffleSplit, cross_val_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

SEED = 42          # fix it once, here, and never silently change it
np.random.seed(SEED)

data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)   # 1 = malignant, same recoding as Week 2
feature_names = list(data.feature_names)
print("X", X.shape, "  y", y.shape)

---

## Part A - Experimental instability

Load the data, define the question, and measure how much a single train/test split's accuracy actually moves around.

## Task 1 — State the question before you touch the data

Week 3 opens with a workflow, not a model. Before writing any code, answer in
the markdown cell below (no code needed for this task):

**Question 1.1.** State, in one sentence each: the biological question, what
one row of `X` is, what the target represents, and who would use a model built
from this workflow.

**Question 1.2.** What is the majority-class baseline for this cohort, and why
must every later accuracy number in this notebook be compared against it
rather than against 50 per cent?

In [ ]:
# No code required. Write your answers in this cell (convert to Markdown).

baseline = None  # compute it, then delete this placeholder line and report it


## Task 2 — One split, one number: how much do you trust it?

Split the data once, stratified, 70/30, `random_state=SEED`. Fit a depth-3
entropy decision tree and report the held-out accuracy.

**Question 2.1.** Is this single number, by itself, evidence that the model
works, or a measurement with an unstated amount of uncertainty attached to it?
Say what you would need in order to tell the difference.

*Before this task, it may help to explore the **Workflow Builder** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 3 — Measure the instability directly

Repeat Task 2's protocol — same model, same test fraction — across 30
different random seeds (0 to 29). Collect the 30 held-out accuracies and report
their mean, standard deviation, minimum and maximum.

**Question 3.1.** How many percentage points separate your best split from
your worst? Is that spread larger or smaller than you expected?

**Question 3.2.** A colleague reports "our model gets 97 per cent." Based on
this experiment, what is your first question for them?

*Before this task, it may help to explore the **Data Splitter** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


---

## Part B - Cross-validation and baselines

Replace one split with five, and make sure every model is judged against a real baseline, not against zero.

## Task 4 — Cross-validation, properly

Run stratified 5-fold cross-validation on the same model. Report the mean, the
standard deviation, and every individual fold score.

**Question 4.1.** Is a train/test split a form of cross-validation? Answer
precisely, using this lab's own vocabulary.

**Question 4.2.** Compare the CV standard deviation with the spread you
measured in Task 3. They are not measuring exactly the same thing — say what
differs.

*Before this task, it may help to explore the **Cross-Validation** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 5 — The unit of splitting: a simulation you build yourself

WDBC has one sample per patient, so this cannot be demonstrated on it directly.
Build a small **illustrative simulation** instead: 40 simulated patients, each
with 2 to 4 samples. Each patient has a latent 6-dimensional "centroid" drawn
from `N(0.5, 1.3)` if their label is 1, `N(-0.5, 1.3)` if 0; each of their
samples is the centroid plus `N(0, 0.2)` noise.

Fit a 1-nearest-neighbour classifier under **(a)** a naive split at the sample
level and **(b)** a group-aware split at the patient level (every sample from
one patient entirely on one side). Report both accuracies.

**Question 5.1.** Name the specific mechanism that lets the naive split score
higher: what is the model actually exploiting?

**Question 5.2.** WDBC itself does not have this problem. Name one dataset
type used elsewhere in this course, or in your own field, where it would.

*Before this task, it may help to explore the **Data Splitter** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here — build the simulated cohort, then compare the two splits


## Task 6 — A baseline ladder

Cross-validate, with the same 5-fold protocol, four models: the majority-class
baseline (`DummyClassifier(strategy="most_frequent")`), the depth-3 tree, an
unpruned tree, and k-NN with k=5 (inside a `Pipeline` with `StandardScaler`,
since k-NN needs scaling).

**Question 6.1.** Which model would you actually recommend, and what did you
weigh besides the accuracy number?

*Before this task, it may help to explore the **Baseline Explorer** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


---

## Part C - Leakage

Build a leak-free pipeline, then deliberately break it and measure what the leak is worth.

## Task 7 — Build the honest pipeline

Build a `Pipeline` of `StandardScaler`, `SelectKBest(f_classif, k=10)` and
`DecisionTreeClassifier(max_depth=3)`. Cross-validate it with the same 5-fold
protocol.

**Question 7.1.** Why does putting the scaler and the selector inside the
`Pipeline`, rather than applying them to `X` beforehand, matter here?

*Before this task, it may help to explore the **Pipeline Builder** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 8 — Break it on purpose, and measure the damage

Now do it wrongly: fit `SelectKBest(k=10)` on the **whole** dataset first, then
cross-validate a tree on the reduced columns. Compare this "leaky" score with
Task 7's honest score.

**Question 8.1.** The gap on WDBC is probably small. Does a small gap mean the
leaky protocol was nearly fine? Explain using what you already know about this
dataset's features.

*Before this task, it may help to explore the **Leakage Simulator** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 9 — The same leak, on data with a known answer

Generate 100 synthetic samples with 5,000 standard-normal features and
**random** labels (`rng.randint(0, 2, 100)`). The true accuracy is exactly 0.5
by construction, because nothing in the features is related to the label.

Run both the leaky and the honest protocol from Tasks 7-8 on this synthetic
data (drop the scaler; it is not needed here).

**Question 9.1.** Report both numbers. What does the leaky protocol's number
actually measure, if not predictive accuracy?

In [ ]:
# your code here


## Task 10 — Duplicate rows

Duplicate 25 per cent of the WDBC rows (concatenate `X`, `y` with a random 25
per cent subsample of themselves) and cross-validate a 1-NN classifier with and
without the duplicates.

**Question 10.1.** Explain the gap in terms of what 1-NN actually does at
prediction time.

In [ ]:
# your code here


---

## Part D - Model selection and reproducibility

Choose a hyperparameter without ever touching the final test set, touch it exactly once, and write down what would let someone else reproduce the result.

## Task 11 — Choose a hyperparameter without touching the test set

Split 80/20 into a train-pool and a final test set, stratified,
`random_state=SEED`. Using 5-fold CV **on the train pool only**, sweep
`max_depth` from 1 to 12 and choose the depth with the best mean CV accuracy.
Do **not** evaluate anything on the final test set yet.

**Question 11.1.** State the depth you chose and its CV accuracy. Then, before
running the next task, write down what you predict the final test accuracy
will be, and how confident you are.

*Before this task, it may help to explore the **Hyperparameter Explorer** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here — do not touch X_final_test / y_final_test below


## Task 12 — Touch the test set exactly once

Fit the chosen-depth tree on the whole train pool and evaluate it on the final
test set, for the first and only time.

**Question 12.1.** Compare training accuracy, pool CV accuracy and final test
accuracy. Which would you put in a report?

*Before this task, it may help to explore the **WDBC Capstone** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 13 — Error analysis

List every false negative (a malignant case the model predicted benign) from
Task 12's final test set, with its `worst perimeter`, `worst concave points`
and `mean texture` values.

**Question 13.1.** Do the missed cases look "typically malignant" on these
features, or borderline? What does that tell you about where this model is
weak?

*Before this task, it may help to explore the **Error Analysis** interactive on the course website (Week 3, Interactive exploration).*

In [ ]:
# your code here


## Task 14 — Sanity-check the workflow order

Below are four short code fragments. For each, say whether it is consistent
with the workflow this lab followed, and if not, name the specific mistake
(use the vocabulary from Task 4, 7 and 11-12: cross-validation, pipeline
leakage, touching the test set).

```python
# Fragment A
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
sc = StandardScaler().fit(X_tr)          # fit on training rows only
X_tr2, X_te2 = sc.transform(X_tr), sc.transform(X_te)

# Fragment B
best_depth, best_score = None, 0
for d in range(1, 15):
    m = DecisionTreeClassifier(max_depth=d).fit(X_tr, y_tr)
    s = m.score(X_te, y_te)              # scored on the test set
    if s > best_score: best_depth, best_score = d, s

# Fragment C
X = StandardScaler().fit_transform(X)    # fit on all of X
scores = cross_val_score(model, X, y, cv=5)

# Fragment D
pipe = Pipeline([("sc", StandardScaler()), ("m", model)])
scores = cross_val_score(pipe, X, y, cv=5)
```

In [ ]:
# No code required. Judge each fragment in the markdown cell below.


## Task 15 — Assemble the evidence summary

Collect the following numbers, computed earlier in this notebook, into a
single table: majority baseline, single-split accuracy (Task 2), 30-split
mean/sd/range (Task 3), 5-fold CV mean/sd (Task 4), honest pipeline CV mean
(Task 7), leaky-vs-honest gap on WDBC (Task 8), leaky-vs-honest gap on pure
noise (Task 9), and final test accuracy (Task 12).

In [ ]:
# your code here — build one small DataFrame from variables you already have


## Task 16 — Reproducibility: write the protocol down

Write a short, model-card-style record of the experiment in Tasks 11-13, in
the markdown cell below. This is the deliverable. Cover:

- **Question and data.** What you were trying to find out, and the dataset.
- **Split.** Sizes, stratification, random_state, exactly which rows never
  touched model selection.
- **Model and hyperparameter search.** What was swept, on what data, and how
  the winner was chosen.
- **Final evaluation.** The one number that counts, and the protocol that
  produced it.
- **Environment.** scikit-learn / numpy / pandas versions (`import sklearn;
  sklearn.__version__` etc.) and the seed.
- **What would break this protocol.** One concrete way someone could
  accidentally leak the test set if they modified this notebook carelessly.

**Be honest about anywhere in this notebook you looked at the test set more
than once** — a model card that describes an idealised process is worse than
none.

*Before this task, it may help to explore the **Reproducibility Record** interactive on the course website (Week 3, Interactive exploration).*

### Protocol record — WDBC decision tree, Tasks 11-13

*(replace this cell with your protocol record)*

**Question and data**

**Split**

**Model and hyperparameter search**

**Final evaluation**

**Environment**

**What would break this protocol**


---

## Before you leave

Check that you can answer these without looking anything up.

1. Why is a single train/test split not the same thing as cross-validation?
2. Name the two places in this notebook a model could see the test set before it should, and what each is called.
3. Why can a small leaky-versus-honest gap NOT be read as proof a protocol was sound?
4. Which number from this notebook goes in a report, and which numbers explain its uncertainty?
